# Red neuronal para tarifas de taxi

Este cuaderno usa `viajes_features.csv` y reutiliza sin modificaciones `dividir()` y `preparacion()` del material de la semana 11.

Arquitectura: **Input (25) -> Hidden 1 (64) -> Hidden 2 (32) -> Hidden 3 (16) -> Output (1)**.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROYECTO = Path.cwd()
STARTER = PROYECTO / 'laboratorios' / 'semana-11-sesion-01' / 'starter'
sys.path.insert(0, str(STARTER))

from preparacion_taxis import dividir, preparacion

viajes = pd.read_csv(STARTER / 'viajes_features.csv')
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = dividir(viajes)

print(f'Viajes: {len(viajes)}')
print(f'Entrenamiento: {X_entrenamiento.shape}; prueba: {X_prueba.shape}')

Viajes: 2975
Entrenamiento: (2380, 10); prueba: (595, 10)


In [2]:
preprocesador = preparacion()
X_entrenamiento_preparado = preprocesador.fit_transform(X_entrenamiento)
X_prueba_preparado = preprocesador.transform(X_prueba)

# El transformador compartido produce 25 columnas: 2 numericas, 5 listas y 18 one-hot.
assert X_entrenamiento_preparado.shape[1] == 25

print('Forma despues de preparar:', X_entrenamiento_preparado.shape)

Forma despues de preparar: (2380, 25)


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor

# MLPRegressor recibe las 25 entradas; sus capas ocultas son 64 -> 32 -> 16 y predice una tarifa.
red = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
)

red.fit(X_entrenamiento_preparado, y_entrenamiento)
predicciones = red.predict(X_prueba_preparado)

mae = mean_absolute_error(y_prueba, predicciones)
rmse = mean_squared_error(y_prueba, predicciones) ** 0.5
print('Arquitectura: 25 -> 64 -> 32 -> 16 -> 1')
print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')

Arquitectura: 25 -> 64 -> 32 -> 16 -> 1
MAE:  1.26
RMSE: 1.67


In [4]:
for weight_layer in red.coefs_:
    print(weight_layer.shape)
    # print(weight_layer)

(25, 64)
[[ 4.20493657e-02  1.13551590e-01  2.09279734e-01 ... -7.18094207e-01
   3.03745208e-01  3.17747702e-02]
 [-6.46450212e-03  3.12913549e-02 -3.59307028e-02 ...  7.99370309e-02
   7.03827581e-02 -1.88498453e-04]
 [-3.44574176e-01  1.06335488e-02 -5.05094718e-02 ... -3.15829612e-02
  -1.59306788e-01  2.74440586e-01]
 ...
 [ 9.05546604e-02  1.48394958e-01  8.79651868e-02 ...  1.82367529e-01
   1.17238899e-01  2.65766279e-01]
 [ 3.11341041e-01  1.89832223e-01  1.49383611e-01 ...  1.36457336e-01
   9.16538101e-02  1.86631192e-02]
 [-1.65611401e-02  1.99111184e-01  3.34534273e-02 ...  1.61464726e-02
  -7.22221920e-02  2.81716876e-01]]
(64, 32)
[[ 0.34305437  0.2655746  -0.15533878 ... -0.083781    0.04308564
  -0.14975663]
 [-0.13900803 -0.08268886  0.213457   ... -0.12211722  0.11541901
   0.35550105]
 [ 0.26918331  0.1680626  -0.07927684 ...  0.11080563 -0.07499995
   0.09011944]
 ...
 [-0.48219649 -0.28577044  0.15244996 ... -0.56925127 -0.4974482
  -0.56304295]
 [ 0.15641273  0.3

In [5]:
for intercept_matrix in red.intercepts_:
    print(intercept_matrix.shape)
    # print(intercept_matrix)

(64,)
(32,)
(16,)
(1,)


In [6]:
print(red.intercepts_[0])

[-0.02093777  0.06198855  0.07614559  0.10832549 -0.09410752  0.17235709
 -0.07444821  0.31161813 -0.07204023  0.18059472  0.1999164   0.25125725
  0.23602323 -0.00908506  0.03463122  0.2494233   0.3195897   0.1653594
  0.39536427  0.16990517  0.12352996 -0.09990127  0.15423678  0.36133887
 -0.0331652   0.26705311 -0.13456242  0.11950132 -0.05635028  0.12902047
  0.27751526  0.01068047  0.21177534  0.31565965  0.44318475  0.04052052
  0.10879871 -0.15336012  0.3748454   0.37384559  0.06166081  0.32311896
  0.36874405  0.24749563 -0.14373327  0.23707246  0.00124183  0.31847828
  0.30991242  0.12058501  0.02733394  0.09831471  0.04098888  0.11685496
  0.00049882  0.38085538  0.07850825  0.0458822   0.15533591  0.12806431
  0.19774608  0.29451471  0.35956888  0.10830261]


In [7]:
red

,loss,'squared_error'
,hidden_layer_sizes,"(64, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,500
,shuffle,True


In [8]:
#? Grab the output up to the first layer and compare value by value with the matrix multiplcation and expanded mathematical linear equation


x_single_case = X_prueba_preparado[0]

#? Now we calculate the
import numpy as np
W1 = red.coefs_[0]
b1 = red.intercepts_[0]

z1: np.ndarray  = x_single_case @ W1 + b1
h1: np.ndarray = np.maximum(0, z1)   # ReLU

In [9]:
print(z1.shape)

(64,)


In [10]:
print(h1.shape)

(64,)


In [17]:
h1

array([0.03696381, 0.41153499, 0.23300519, 0.72967963, 0.        ,
       0.62160699, 0.09027119, 0.28043676, 0.        , 0.62872809,
       0.54559468, 0.10516569, 0.05792271, 0.36463237, 0.4435874 ,
       0.20765384, 1.12607585, 0.92483037, 0.53092331, 0.42814308,
       0.32670888, 0.60966622, 0.22421213, 0.68013982, 0.85748317,
       0.0341278 , 0.27194206, 0.382474  , 0.53342222, 0.24673117,
       0.72058328, 0.        , 1.04542527, 0.27904758, 0.19115096,
       0.2116144 , 0.84089939, 0.04539313, 0.10947405, 0.41315401,
       0.        , 0.893209  , 1.3658746 , 0.36036598, 0.        ,
       0.27733486, 0.20649379, 0.73546516, 0.32405186, 0.        ,
       0.37897526, 0.82659696, 0.54731998, 0.39753558, 0.53937781,
       0.7573846 , 0.15037563, 0.        , 0.58908033, 0.37843362,
       0.32114376, 0.92278261, 0.33690344, 0.07050617])

In [25]:
z1

array([ 0.03696381,  0.41153499,  0.23300519,  0.72967963, -0.2017119 ,
        0.62160699,  0.09027119,  0.28043676, -0.39894267,  0.62872809,
        0.54559468,  0.10516569,  0.05792271,  0.36463237,  0.4435874 ,
        0.20765384,  1.12607585,  0.92483037,  0.53092331,  0.42814308,
        0.32670888,  0.60966622,  0.22421213,  0.68013982,  0.85748317,
        0.0341278 ,  0.27194206,  0.382474  ,  0.53342222,  0.24673117,
        0.72058328, -0.13237795,  1.04542527,  0.27904758,  0.19115096,
        0.2116144 ,  0.84089939,  0.04539313,  0.10947405,  0.41315401,
       -0.06674329,  0.893209  ,  1.3658746 ,  0.36036598, -0.3593551 ,
        0.27733486,  0.20649379,  0.73546516,  0.32405186, -0.48560238,
        0.37897526,  0.82659696,  0.54731998,  0.39753558,  0.53937781,
        0.7573846 ,  0.15037563, -0.08108688,  0.58908033,  0.37843362,
        0.32114376,  0.92278261,  0.33690344,  0.07050617])